# Tutorial: Milestone 0 Contract Freeze + Parity Harness

Audience:
- Engineers validating Phase 1 API contract parity during Python to Rust migration.

Prerequisites:
- Run this notebook from the repository root.
- `uv` and `cargo` are installed.

Learning goals:
- Verify canonical parity fixtures exist and load.
- Run the parity harness against Python backend.
- Regenerate fixtures in guarded mode.
- Confirm Rust daemon baseline tests are green.


In [2]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate repository root (no pyproject.toml in current path or parents).")

FIXTURE_PATH = REPO_ROOT / "tests/parity/fixtures/phase1/corpus.json"
print(f"repo: {REPO_ROOT}")
print(f"fixture path: {FIXTURE_PATH}")


repo: /Users/austin/GitHub/lucida
fixture path: /Users/austin/GitHub/lucida/tests/parity/fixtures/phase1/corpus.json


## Step 1 - Verify Python parity harness gate

Expected result: command exits 0 and reports `1 passed`.


In [3]:
result = subprocess.run(
    ["uv", "run", "pytest", "tests/test_phase1_parity.py", "-q"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Parity gate failed."


.                                                                        [100%]
1 passed in 0.14s



## Step 2 - Inspect canonical fixture corpus

Expected result: fixture file exists and includes cases for all Phase 1 endpoints.


In [4]:
payload = json.loads(FIXTURE_PATH.read_text(encoding="utf-8"))
cases = payload["cases"]
names = [item["name"] for item in cases]
print(f"fixture schema_version: {payload['schema_version']}")
print(f"total cases: {len(cases)}")
print("first 8 cases:", names[:8])
required_prefixes = [
    "dataset_open_",
    "session_create_",
    "view_create_",
    "view_get_",
    "view_update_",
    "export_viewstate_",
    "import_viewstate_",
    "render_image_",
]
for prefix in required_prefixes:
    assert any(name.startswith(prefix) for name in names), f"missing fixture group: {prefix}"


fixture schema_version: 1
total cases: 29
first 8 cases: ['dataset_open_success', 'dataset_open_invalid_metadata_error', 'dataset_open_invalid_request_error', 'session_create_success', 'dataset_open_unknown_session_error', 'dataset_open_with_session_success', 'view_create_success', 'view_get_success']


## Step 3 - Regenerate fixtures in guarded mode

Expected result: regeneration command exits 0 and does not change tracked fixtures when behavior is unchanged.


In [5]:
regen_env = dict(os.environ)
regen_env["LUCIDA_REGEN_PARITY_FIXTURES"] = "1"
regen = subprocess.run(
    ["uv", "run", "pytest", "tests/test_phase1_parity.py", "-q"],
    cwd=REPO_ROOT,
    env=regen_env,
    capture_output=True,
    text=True,
)
print(regen.stdout)
if regen.stderr:
    print(regen.stderr)
assert regen.returncode == 0, "Fixture regeneration failed."
diff = subprocess.run(
    ["git", "diff", "--", str(FIXTURE_PATH)],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
assert diff.stdout.strip() == "", "Fixture corpus changed after regeneration. Inspect diff."
print("fixture corpus stable after regeneration")


.                                                                        [100%]
1 passed in 0.13s

fixture corpus stable after regeneration


## Step 4 - Validate Rust daemon baseline

Expected result: `cargo test` exits 0 for the new workspace/crate scaffold.


In [6]:
cargo = subprocess.run(
    ["cargo", "test"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(cargo.stdout)
if cargo.stderr:
    print(cargo.stderr)
assert cargo.returncode == 0, "cargo test failed."



running 0 tests

test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s


running 0 tests

test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s


running 2 tests
test error_envelope_serializes_shape ... ok
test api_error_into_response_uses_status_and_envelope ... ok

test result: ok. 2 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s


running 1 test
test healthz_returns_ok_status_payload ... ok

test result: ok. 1 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s


running 0 tests

test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s


    Finished `test` profile [unoptimized + debuginfo] target(s) in 0.10s
     Running unittests src/lib.rs (target/debug/deps/lucida_daemon-775cca63779a964d)
     Running unittests src/main.rs (target/debug/deps/lucida_daemon-ba5bf1ddd0625cfd)
     Running tests/error_envelope.rs (ta

## Milestone 0 done

If all assertions passed, Milestone 0 gates are satisfied for:
- Python parity fixture corpus + harness validation.
- Guarded fixture regeneration workflow.
- Rust daemon scaffold baseline tests.
